# Qwen3 teacher 규모 비교

공개 Cartridges Qwen benchmark 예제의 학습·평가 설정을 사용한다.

| 조건 | 생성 모델 | Scoring teacher | Student |
|---|---|---|---|
| A | Qwen3-4B | Qwen3-4B | Qwen3-4B |
| B | Qwen3-4B | Qwen3-8B | Qwen3-4B |
| C | Qwen3-8B | Qwen3-4B | Qwen3-4B |
| D | Qwen3-8B | Qwen3-8B | Qwen3-4B |

A/B와 C/D는 각각 같은 대화를 사용한다. 주 비교는 B−A다.
LongHealth: 200문항 정확도. MTOB: Kalamang → English 50문장 chrF.

실행 환경: Linux x86_64, Python 3.12, CUDA BF16 GPU. GPU 전체 경로는 검증 전이다.

Colab의 **런타임 → 런타임 유형 변경**에서 런타임 버전 `2026.07`(Python 3.12)과 BF16을 지원하는 GPU를 선택한다. 실행 설정을 선택한 뒤 셀을 위에서 아래로 실행한다.


## 1. 실행 설정

처음에는 `MODE=train`, `PROFILE=smoke`로 전체 경로를 확인한다.

| 설정 | 동작 |
|---|---|
| `CONDITION=A/B/C/D` | 선택한 조건 하나 실행 (`all`은 네 조건 실행) |
| `MODE=train` | 데이터 준비 → 합성·scoring → 카트리지 학습·평가 |
| `MODE=evaluate` | 저장된 checkpoint와 최종 평가 RNG 상태로 재평가 |
| `PROFILE=smoke` | 32대화·64-token cache·최대 2 steps, 평가 2문항 |
| `PROFILE=pilot` | 512대화·최대 16 steps |
| `PROFILE=main` | 생성 모델당 131,072대화, 공개 예제의 epoch |

조건을 나누어 실행할 때는 같은 `RUN_NAME`·benchmark·profile을 유지하고 `CONDITION`을 바꾼다. A/B는 G4 대화, C/D는 G8 대화를 재사용한다.

재평가에는 기존 실행과 같은 `RUN_NAME`, `BENCHMARK`, `PROFILE`, 학습 seed를 사용한다. `SEEDS`가 비어 있으면 main은 42·123·2026, smoke·pilot은 42를 사용한다. `USE_DRIVE=True`이면 실행 폴더를 Drive에 저장하고 새 런타임에서 복원한다.


In [ ]:
# @title 실행 설정
CONDITION = 'A'  # @param ['A', 'B', 'C', 'D', 'all']
MODE = 'train'  # @param ['train', 'evaluate']
BENCHMARK = 'longhealth'  # @param ['longhealth', 'mtob']
PROFILE = 'smoke'  # @param ['smoke', 'pilot', 'main']
RUN_NAME = 'qwen3-v1-smoke-001'  # @param {type:'string'}
SEEDS = ''  # @param {type:'string'}
USE_DRIVE = True  # @param {type:'boolean'}


## 2. 소스와 환경

고정된 Git commit의 실험 소스를 자동으로 내려받는다.


In [ ]:
# @title 실험 소스 준비
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from IPython.display import display

SOURCE_REPO = 'https://github.com/3ae3ae/qwen3-teacher-scaling.git'
SOURCE_REVISION = '1d387eba088dae6f0b6fab99e0bcc9bd700f44e4'
ROOT = Path('/content/cartridge-teacher-scaling')
if not ROOT.exists():
    subprocess.run(['git', 'init', str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'remote', 'add', 'origin', SOURCE_REPO], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', SOURCE_REVISION], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if revision != SOURCE_REVISION:
    raise ValueError('기존 소스의 commit이 다릅니다. 새 런타임에서 실행하세요.')
subprocess.run(['git', '-C', str(ROOT), 'diff', '--exit-code', 'HEAD', '--'], check=True)
print('Source commit:', revision)


Tokasaurus의 PyTorch 2.6.0·Transformers 4.53.0을 공통 환경으로 사용한다. Python 패키지는 requirements.lock의 버전 목록에서 설치한다.


In [ ]:
# @title 환경 설치와 연결 검사
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Python 3.12 런타임이 필요합니다.')
PYTHON = ROOT / '.venv/bin/python'
if not (ROOT / '.venv/bin/pip').exists():
    subprocess.run(['uv', 'venv', '--allow-existing', '--seed', '--python', sys.executable, str(ROOT / '.venv')], check=True)
for script in ['scripts/setup.py', 'tests/check.py']:
    with subprocess.Popen([str(PYTHON), '-u', str(ROOT / script)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='')
        if process.wait():
            raise RuntimeError(f'{script} 실패: 위 로그를 확인하세요.')


## 3. 실행 폴더

설정과 소스 버전을 확인하고 저장된 실행을 복원한다. 완료된 준비·합성·scoring·학습 단계는 재사용한다. 중단된 학습을 처음부터 다시 시작할 때는 새 `RUN_NAME`을 사용한다.


In [ ]:
# @title 실행 폴더 준비
BASE = json.loads((ROOT / 'configs/experiment.json').read_text())
CONDITIONS = list(BASE['conditions']) if CONDITION == 'all' else [CONDITION]
if any(condition not in BASE['conditions'] for condition in CONDITIONS):
    raise ValueError('A, B, C, D 또는 all을 선택하세요.')
TRAIN_SEEDS = list(dict.fromkeys(int(x.strip()) for x in SEEDS.split(','))) if SEEDS.strip() else (BASE['seeds_for_main'] if PROFILE == 'main' else [BASE['seed']])
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('실행 이름에는 영문·숫자·밑줄·하이픈을 사용하세요.')
RUN = Path('/content/runs') / RUN_NAME / BENCHMARK
BACKUP = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/cartridge-teacher-scaling') / RUN_NAME / BENCHMARK
    if BACKUP.exists() and not RUN.exists():
        shutil.copytree(BACKUP, RUN)
if MODE == 'evaluate':
    required = [RUN / 'study.json', RUN / 'prepare.summary.json']
    required.extend(RUN / f'seed-{seed}/{condition}/cache_last.pt'
                    for seed in TRAIN_SEEDS for condition in CONDITIONS)
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError('저장된 실행을 확인하세요: ' + ', '.join(missing))
RUN.mkdir(parents=True, exist_ok=True)
CODE_SHA = hashlib.sha256((ROOT / 'experiment.py').read_bytes()).hexdigest()
PATCH_SHA = hashlib.sha256((ROOT / 'patches/cartridges.patch').read_bytes()).hexdigest()
LOCK_SHA = hashlib.sha256((ROOT / 'requirements.lock').read_bytes()).hexdigest()
STUDY = {'source_revision': SOURCE_REVISION, 'lock_sha256': LOCK_SHA, 'benchmark': BENCHMARK, 'profile': PROFILE, 'config': BASE, 'code_sha256': CODE_SHA, 'patch_sha256': PATCH_SHA}
if (RUN / 'study.json').exists():
    previous = json.loads((RUN / 'study.json').read_text())
    if any(previous.get(key) != value for key, value in STUDY.items() if key != 'source_revision'):
        raise ValueError('기존 실행과 설정이 다릅니다. 새 RUN_NAME을 사용하세요.')
else:
    (RUN / 'study.json').write_text(json.dumps(STUDY, indent=2))
print(f'Mode: {MODE}, benchmark: {BENCHMARK}, profile: {PROFILE}, seeds: {TRAIN_SEEDS}')
print('Conditions:', CONDITIONS)
print('Results:', RUN)
print('Drive:', BACKUP)


In [ ]:
# @title 단계 실행 함수
def run_stage(stage, *, generator='4B', teacher='4B', condition='A', seed=42):
    if stage == 'prepare':
        summary = RUN / 'prepare.summary.json'
    elif stage == 'synthesize':
        summary = RUN / f'G{generator}/synthesize.summary.json'
    elif stage == 'score':
        summary = RUN / f'G{generator}/score-{teacher}.summary.json'
    else:
        summary = RUN / f'seed-{seed}/{stage}-{condition}.summary.json'
    if summary.exists():
        result = json.loads(summary.read_text())
        if result['code_sha256'] != CODE_SHA or result['patch_sha256'] != PATCH_SHA or result.get('lock_sha256') != LOCK_SHA:
            raise ValueError('기존 결과와 코드가 다릅니다.')
        if stage != 'evaluate':
            print(f'{stage}: 저장된 결과 사용')
            return result
    summary.parent.mkdir(parents=True, exist_ok=True)
    command = [str(PYTHON), '-u', str(ROOT / 'experiment.py'), stage,
               '--benchmark', BENCHMARK, '--profile', PROFILE, '--out', str(RUN),
               '--generator', generator, '--teacher', teacher, '--condition', condition, '--seed', str(seed)]
    with summary.with_suffix('.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='')
        status = process.wait()
    if BACKUP:
        shutil.copytree(RUN, BACKUP, dirs_exist_ok=True, ignore=shutil.ignore_patterns('models'))
    if status:
        raise RuntimeError(f'{stage} 실패: {summary.with_suffix(".log")}')
    return json.loads(summary.read_text())


## 4. 데이터

공개 resource sampler에서 batch당 발췌 하나와 32개 seed prompt를 준비한다. 재평가는 저장된 데이터를 사용한다.


In [ ]:
# @title 데이터 준비
if MODE == 'train':
    run_stage('prepare')


## 5. 합성과 scoring

학습 모드에서 공개 Tokasaurus client와 SelfStudySynthesizer로 대화를 생성하고, 선택한 조건의 teacher가 저장된 답변의 분포를 계산한다. 서버는 생성 단계가 끝나면 종료된다.


In [ ]:
# @title 합성과 scoring
if MODE == 'train':
    for generator in dict.fromkeys(BASE['conditions'][c]['generator'] for c in CONDITIONS):
        run_stage('synthesize', generator=generator)
        for condition in CONDITIONS:
            pair = BASE['conditions'][condition]
            if pair['generator'] == generator:
                display(run_stage('score', generator=generator, teacher=pair['teacher']))


## 6. 학습과 평가

`train`은 공개 TrainConfig의 초기화·loss·패킹·optimizer·주기적 평가를 사용한다. `evaluate`는 조건·seed별 최종 checkpoint를 불러와 평가한다.


In [ ]:
# @title 학습 또는 재평가
for seed in TRAIN_SEEDS:
    for condition in CONDITIONS:
        display(run_stage(MODE, condition=condition, seed=seed))


## 7. 결과

아래 표는 같은 실행 폴더에서 완료된 조건의 점수와 조건별 평균·표준편차를 보여준다. A와 B의 동일 seed 결과가 있으면 B−A를 표시한다.

결과는 `/content/runs/<RUN_NAME>/<BENCHMARK>`에 저장된다. `USE_DRIVE=True`이면 Drive의 `cartridge-teacher-scaling/<RUN_NAME>/<BENCHMARK>`에도 복사된다.

실행 폴더에는 config·환경 정보, 생성 대화·soft targets, 초기 cache, checkpoint·prediction·metric·평가 RNG 상태가 저장된다. Prediction에는 평가 질문과 정답이 포함되며 해당 데이터의 라이선스가 적용된다.


In [ ]:
# @title 평가 결과
import pandas as pd
completed = []
for seed in TRAIN_SEEDS:
    for condition in BASE['conditions']:
        path = RUN / f'seed-{seed}/{MODE}-{condition}.summary.json'
        if path.exists():
            saved = json.loads(path.read_text())
            metric = saved['evaluations'][-1] if MODE == 'train' else saved
            completed.append({'condition': condition, 'seed': seed,
                              **{key: metric[key] for key in ['questions', 'accuracy', 'missing_answer_tags', 'chrf'] if key in metric}})
results = pd.DataFrame(completed)
metric_name = 'accuracy' if BENCHMARK == 'longhealth' else 'chrf'
display(results)
display(results.groupby('condition')[metric_name].agg(['mean', 'std', 'count']))
paired = results.pivot(index='seed', columns='condition', values=metric_name)
if {'A', 'B'}.issubset(paired.columns):
    display((paired['B'] - paired['A']).dropna().rename('B_minus_A').to_frame())
results.to_csv(RUN / f'{MODE}-metrics.csv', index=False)
if BACKUP:
    shutil.copy2(RUN / f'{MODE}-metrics.csv', BACKUP / f'{MODE}-metrics.csv')
